## Example 1: Basic HPACK Encoder usage

In [ ]:
import hpack

# Example 1: Basic HPACK Encoder usage
print("Example 1: Basic HPACK Encoder")

# Create an HPACK encoder
encoder = hpack.Encoder()

print(f"Encoder type: {type(encoder).__name__}")
print(f"Initial header table size: {encoder.header_table_size}")

# Encode some HTTP headers
headers = [
    (':method', 'GET'),
    (':path', '/index.html'),
    (':authority', 'example.com'),
    ('user-agent', 'Python-HPACK/1.0'),
    ('accept', 'text/html')
]

# Encode the headers
encoded_data = encoder.encode(headers)

print(f"Original headers: {len(headers)} items")
print(f"Encoded data length: {len(encoded_data)} bytes")
print(f"Encoded data (hex): {encoded_data[:20].hex()}...")

# Show header table after encoding
print(f"Header table type: {type(encoder.header_table).__name__}")
print(f"Dynamic table size: {encoder.header_table_size}")

# The static table already covers :method GET, so that header costs a single byte (0x82)
assert encoded_data[0] == 0x82
assert len(encoded_data) < sum(len(n) + len(v) for n, v in headers)
assert hpack.Decoder().decode(encoded_data) == headers
assert encoder.header_table_size == 4096  # the HTTP/2 default

## Example 2: Basic HPACK Decoder usage

In [ ]:
import hpack

# Example 2: Basic HPACK Decoder usage  
print("Example 2: Basic HPACK Decoder")

# Create an HPACK decoder
decoder = hpack.Decoder()

print(f"Decoder type: {type(decoder).__name__}")
print(f"Initial header table size: {decoder.header_table_size}")

# Create some encoded data (simulate received data)
encoder = hpack.Encoder()
original_headers = [
    (':method', 'POST'),
    (':path', '/api/data'),
    (':authority', 'api.example.com'),
    ('content-type', 'application/json'),
    ('content-length', '123')
]

encoded_data = encoder.encode(original_headers)
print(f"Encoded data to decode: {len(encoded_data)} bytes")

# Decode the headers
decoded_headers = decoder.decode(encoded_data)

print(f"Decoded headers: {len(decoded_headers)} items")
for name, value in decoded_headers:
    print(f"  {name}: {value}")

print(f"Header table type: {type(decoder.header_table).__name__}")
print(f"Dynamic table size: {decoder.header_table_size}")

assert decoded_headers == original_headers
assert all(isinstance(h, hpack.HeaderTuple) for h in decoded_headers)

## Example 3: Header table management and compression demonstration

In [ ]:
import hpack

# Example 3: Header table management and compression demonstration
print("Example 3: Header table management")

# Create encoder for header compression
encoder = hpack.Encoder()

print(f"Initial table size: {encoder.header_table_size}")

# Add headers to populate the dynamic table
headers_batch1 = [
    ('custom-header-1', 'value1'),
    ('custom-header-2', 'value2'),
    ('custom-header-3', 'value3')
]

encoded1 = encoder.encode(headers_batch1)
print(f"After batch 1 - table size: {encoder.header_table_size}")
print(f"Batch 1 encoded size: {len(encoded1)} bytes")

# Add more headers with some repetition to show compression
headers_batch2 = [
    ('custom-header-1', 'value1'),  # Reuse existing header
    ('custom-header-4', 'value4'),  # New header
    ('custom-header-2', 'value2'),  # Reuse existing header
]

encoded2 = encoder.encode(headers_batch2)
print(f"After batch 2 - table size: {encoder.header_table_size}")
print(f"Batch 2 encoded size: {len(encoded2)} bytes")

# Demonstrate compression efficiency
total_original = len(str(headers_batch1)) + len(str(headers_batch2))
total_encoded = len(encoded1) + len(encoded2)
print(f"Compression ratio: {total_encoded}/{total_original} = {total_encoded/total_original:.2f}")

# Repeated name/value pairs are sent as one-byte references into the dynamic table
encoded3 = encoder.encode(headers_batch1)
print(f"Batch 1 again: {len(encoded3)} bytes")
assert len(encoded3) == len(headers_batch1) < len(encoded1)
assert len(encoded2) < len(encoded1)  # batch 2 reuses two of its three headers

## Example 4: HeaderTuple operations and header encoding

In [ ]:
import hpack

# Example 4: HeaderTuple operations and header encoding
print("Example 4: HeaderTuple operations")

# Create HeaderTuple instances
header1 = hpack.HeaderTuple('authorization', 'Bearer token123')
header2 = hpack.HeaderTuple('cookie', 'session=abc123')

print(f"Header 1: {header1}")
print(f"Header 2: {header2}")
# HeaderTuple behaves like a tuple, access by index
print(f"Header 1 name: {header1[0]}")
print(f"Header 1 value: {header1[1]}")
print(f"Header 2 name: {header2[0]}")
print(f"Header 2 value: {header2[1]}")

# Create headers list with different types
headers = [
    header1,
    header2,
    ('content-type', 'application/json'),  # Regular tuple
    hpack.HeaderTuple('x-api-key', 'secret-key')
]

# Encode headers
encoder = hpack.Encoder()
encoded_data = encoder.encode(headers)

print(f"Encoded {len(headers)} headers")
print(f"Encoded data length: {len(encoded_data)} bytes")
print(f"Dynamic table size: {encoder.header_table_size}")

assert header1 == ("authorization", "Bearer token123") and header1.indexable
# NeverIndexedHeaderTuple marks a value (e.g. a credential) that must never enter a compression table
secret = hpack.NeverIndexedHeaderTuple("authorization", "Bearer token123")
assert not secret.indexable
decoded = hpack.Decoder().decode(hpack.Encoder().encode([secret, ("accept", "*/*")]))
assert decoded == [("authorization", "Bearer token123"), ("accept", "*/*")]
assert isinstance(decoded[0], hpack.NeverIndexedHeaderTuple)

## Example 5: Complete encode/decode cycle with compression

In [ ]:
import hpack

# Example 5: Complete encode/decode cycle with compression
print("Example 5: Complete encode/decode cycle")

# Create matched encoder/decoder pair  
encoder = hpack.Encoder()
decoder = hpack.Decoder()

# Test multiple header sets to show compression benefits
request1_headers = [
    (':method', 'GET'),
    (':path', '/api/users'),
    (':authority', 'api.example.com'), 
    ('user-agent', 'MyApp/1.0'),
    ('accept', 'application/json')
]

request2_headers = [
    (':method', 'GET'), 
    (':path', '/api/posts'),  # Different path
    (':authority', 'api.example.com'),  # Same authority
    ('user-agent', 'MyApp/1.0'),  # Same user-agent
    ('accept', 'application/json')  # Same accept
]

# Encode both requests
encoded1 = encoder.encode(request1_headers)
encoded2 = encoder.encode(request2_headers)

print(f"Request 1 encoded: {len(encoded1)} bytes")
print(f"Request 2 encoded: {len(encoded2)} bytes (should be smaller due to compression)")

# Decode both requests
decoded1 = decoder.decode(encoded1)
decoded2 = decoder.decode(encoded2)

print(f"Decoded request 1: {len(decoded1)} headers")
print(f"Decoded request 2: {len(decoded2)} headers")

# Verify round-trip accuracy
assert decoded1 == request1_headers
assert decoded2 == request2_headers
assert len(encoded2) < len(encoded1)  # only :path changed, the rest comes from the dynamic table
print("Both requests round-tripped")